In [ ]:
!pip install --upgrade snowflake-connector-python pandas groq
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sour

In [2]:
import snowflake.connector
import pandas as pd
from google.colab import userdata
import logging


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def get_snowflake_connection():
    """
    Establishes a connection to Snowflake using credentials stored in Colab Secrets.
    """
    try:
        conn = snowflake.connector.connect(
            user=userdata.get('SNOWFLAKE_USERNAME'),
            password=userdata.get('SNOWFLAKE_PASSWORD'),
            account=userdata.get('SNOWFLAKE_ACCOUNT'),
            warehouse=userdata.get('SNOWFLAKE_WAREHOUSE'),
            database=userdata.get('SNOWFLAKE_DATABASE'),
            schema=userdata.get('SNOWFLAKE_SCHEMA')
        )
        logging.info("Successfully connected to Snowflake.")
        return conn
    except Exception as e:
        logging.error(f"Failed to connect to Snowflake: {e}")
        return None


conn = get_snowflake_connection()
if conn:
    cursor = conn.cursor()
    cursor.execute("SELECT CURRENT_VERSION(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()")
    row = cursor.fetchone()
    print(f"\nConnection Success!")
    print(f"Snowflake Version: {row[0]}")
    print(f"Active Warehouse: {row[1]}")
    print(f"Active Database: {row[2]}")
    conn.close()
else:
    print("\nConnection Failed. Please check your Colab Secrets.")


Connection Success!
Snowflake Version: 10.20.102
Active Warehouse: TRANSFORMING
Active Database: FINANCEDB


In [3]:
def fetch_metadata(connection):
    """
    Retrieves metadata for MART and the INTER master table.
    Excludes the RAW schema per user request.
    """
    metadata_query = """
    SELECT
        table_schema,
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_catalog = CURRENT_DATABASE()
      AND table_schema NOT IN ('INFORMATION_SCHEMA', 'RAW')
    ORDER BY
        CASE
            WHEN table_name = 'INTER' AND table_schema = 'INTER' THEN 1
            WHEN table_name LIKE '%MART%' THEN 2
            ELSE 3
        END,
        table_name,
        ordinal_position;
    """
    try:
        cursor = connection.cursor()
        cursor.execute(metadata_query)
        df = cursor.fetch_pandas_all()
        if df.empty:
            logging.warning(f"No metadata found in {connection.database}. Check permissions.")
        else:
            logging.info(f"Metadata found: {len(df)} columns across {df['TABLE_SCHEMA'].nunique()} schemas.")
        return df
    except Exception as e:
        logging.error(f"Error fetching metadata: {e}")
        return None

In [4]:
def generate_schema_context(metadata_df):
    """
    Converts metadata into a format that tells the LLM the fully qualified name (SCHEMA.TABLE).
    """
    if metadata_df is None or metadata_df.empty:
        return "No schema metadata available. Please check Snowflake permissions."

    schema_text = "Available Tables (Use Fully Qualified Names Schema.Table):\n"
    current_table = ""

    for _, row in metadata_df.iterrows():
        full_table_name = f"{row['TABLE_SCHEMA']}.{row['TABLE_NAME']}"
        if full_table_name != current_table:
            current_table = full_table_name
            schema_text += f"\nTable: {current_table}\nColumns:\n"
        schema_text += f"  - {row['COLUMN_NAME']} ({row['DATA_TYPE']})\n"

    return schema_text

In [5]:
from groq import Groq
import re

def get_groq_client():
    return Groq(api_key=userdata.get('GROQ_API_KEY'))

def validate_sql(sql_query):
    forbidden = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "TRUNCATE", "MERGE"]
    query_upper = sql_query.upper()
    if not re.search(r'^\s*(SELECT|WITH)', query_upper):
        return False, "Only SELECT queries are allowed."
    for kw in forbidden:
        if re.search(rf"\b{kw}\b", query_upper):
            return False, f"Unsafe SQL: {kw} is blocked."
    return True, "Safe"

def execute_query(sql_query, connection):
    try:
        cursor = connection.cursor()
        cursor.execute(sql_query)
        df = cursor.fetch_pandas_all()
        logging.info(f"Query executed. Returned {len(df)} rows.")
        return df
    except Exception as e:
        logging.error(f"Execution Error: {e}")
        return pd.DataFrame({"Error": [str(e)]})

def generate_sql(question, schema_context):
    client = get_groq_client()
    system_prompt = f"""
    You are a Snowflake SQL Expert.
    SCHEMA CONTEXT:
    {schema_context}

    RULES:
    1. Use fully qualified names (SCHEMA.TABLE_NAME).
    2. The table INTER.INTER is the Master Table containing all raw details.
    3. Use MART tables for summary questions, but use INTER.INTER for complex relationships or drill-downs.
    4. Return ONLY the raw SQL code. No markdown.
    """
    models = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"]

    for model_name in models:
        try:
            completion = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}],
                temperature=0
            )
            sql = completion.choices[0].message.content.strip().replace("```sql", "").replace("```", "").strip()
            is_safe, msg = validate_sql(sql)
            if is_safe:
                return sql
        except Exception as e:
            logging.warning(f"Error with {model_name}: {e}")
            continue
    return None

In [6]:
def generate_insights(question, df):
    if df is None or df.empty or "Error" in df.columns:
        return "No data found or an error occurred during query execution."

    client = get_groq_client()
    data_summary = df.to_string(index=False)
    system_prompt = "You are a Senior BI Analyst. Explain these Snowflake results and provide strategic recommendations."
    user_prompt = f"Question: {question}\nData Results:\n{data_summary}"

    models = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant"]

    for model_name in models:
        try:
            completion = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
                temperature=0.5
            )
            return completion.choices[0].message.content
        except Exception as e:
            logging.warning(f"Insight Generation Error with {model_name}: {e}")
            continue
    return "Could not generate insights at this time due to API limits or model errors."

In [7]:
def generate_schema_context(metadata_df):
    """
    Converts metadata into a format that tells the LLM the fully qualified name (SCHEMA.TABLE).
    """
    if metadata_df is None or metadata_df.empty:
        return "No schema metadata available. Please check Snowflake permissions."

    schema_text = "Available Tables (Use Fully Qualified Names Schema.Table):\n"
    current_table = ""

    for _, row in metadata_df.iterrows():
        full_table_name = f"{row['TABLE_SCHEMA']}.{row['TABLE_NAME']}"
        if full_table_name != current_table:
            current_table = full_table_name
            schema_text += f"\nTable: {current_table}\nColumns:\n"
        schema_text += f"  - {row['COLUMN_NAME']} ({row['DATA_TYPE']})\n"

    return schema_text

def answer_question(question):
    """
    Phase 7: End-to-end orchestration.
    Returns both the insight text and the underlying DataFrame.
    """
    logging.info(f"User Inquiry: {question}")

    # 1. Get current schema context
    conn = get_snowflake_connection()
    if not conn:
        return "Connection to Snowflake failed.", None

    try:
        metadata_df = fetch_metadata(conn)
        schema_ctx = generate_schema_context(metadata_df)

        # 2. Generate SQL
        sql = generate_sql(question, schema_ctx)
        if not sql:
            return "I was unable to generate a safe or valid SQL query for that question.", None

        print(f"\n[DEBUG] Generated SQL:\n{sql}\n")

        # 3. Execute Query
        df = execute_query(sql, conn)

        # 4. Generate Insights
        insights = generate_insights(question, df)

        return insights, df
    finally:
        conn.close()

In [8]:
user_input = input("Ask a business question: ")
if user_input.strip():
    insights, df = answer_question(user_input)

    print("\n--- Data Table ---")
    if df is not None and not df.empty:
        display(df)
    else:
        print("No data returned.")

    print("\n--- AI Assistant Insights ---")
    print(insights)
else:
    print("Please enter a valid question.")

Ask a business question: which currency is used at most?

[DEBUG] Generated SQL:
SELECT CURRENCY FROM MARTS.MART_CURRENCY ORDER BY NUMBER_OF_TRANSACTIONS DESC LIMIT 1


--- Data Table ---


,CURRENCY
0,USD



--- AI Assistant Insights ---
**Analysis:**

The data results indicate that the currency used most frequently is the **United States Dollar (USD)**.

**Insight:**

This suggests that the majority of transactions or business operations are conducted in USD, which could be due to various factors such as:

* The company's primary market or customer base is in the United States
* The industry or sector is dominated by US-based companies or transactions
* The company has a significant presence in the US market or has a large number of US-based customers

**Strategic Recommendations:**

1. **Optimize pricing and revenue strategies**: Given the dominance of USD, it's essential to review pricing strategies to ensure they are competitive and aligned with the US market.
2. **Streamline currency conversion processes**: Although USD is the primary currency, it's crucial to have efficient currency conversion processes in place to accommodate transactions in other currencies.
3. **Monitor exchange 